In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi

Thu Apr  2 17:49:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install spacy-transformers
!pip install spacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled tra

In [ ]:
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
import json

In [5]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 92.8 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import train_test_split
import sys, fitz

In [ ]:
cv_data = json.load(open('/content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/train_data.json' , 'r'))
len(cv_data)
cv_data[0]

['Govardhana K Senior Software Engineer  Bengaluru, Karnataka, Karnataka - Email me on Indeed: indeed.com/r/Govardhana-K/ b2de315d95905b68  Total IT experience 5 Years 6 Months Cloud Lending Solutions INC 4 Month • Salesforce Developer Oracle 5 Years 2 Month • Core Java Developer Languages Core Java, Go Lang Oracle PL-SQL programming, Sales Force Developer with APEX.  Designations & Promotions  Willing to relocate: Anywhere  WORK EXPERIENCE  Senior Software Engineer  Cloud Lending Solutions -  Bangalore, Karnataka -  January 2018 to Present  Present  Senior Consultant  Oracle -  Bangalore, Karnataka -  November 2016 to December 2017  Staff Consultant  Oracle -  Bangalore, Karnataka -  January 2014 to October 2016  Associate Consultant  Oracle -  Bangalore, Karnataka -  November 2012 to December 2013  EDUCATION  B.E in Computer Science Engineering  Adithya Institute of Technology -  Tamil Nadu  September 2008 to June 2012  https://www.indeed.com/r/Govardhana-K/b2de315d95905b68?isid=rex-

In [ ]:
!python -m spacy init fill-config /content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/base_config.cfg /content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/config.cfg

✔ Auto-filled config with all values
✔ Saved config
/content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
def get_spacy_docs(file, data):
  nlp = spacy.blank('en')
  db = DocBin()

  for text, annot in tqdm(data):
    doc = nlp.make_doc(text)
    annot = annot['entities']

    ents = []
    entity_indices = []

    for start, end, label in annot:
      skip_entity = False
      for idx in range(start, end):
        if idx in entity_indices:
          skip_entity = True
          break
      if skip_entity == True:
        continue

      entity_indices = entity_indices + list(range(start, end))

      try:
        span = doc.char_span(start, end, label=label, alignment_mode='strict')
      except:
          continue

      if span is None:
        err_data = str([start, end]) + "    " + str(text)+ "\n"
        file.write(err_data)

      else:
        ents.append(span)

    try:
      doc.ents = ents
      db.add(doc)
    except:
      pass

  return db

In [ ]:
from sklearn.model_selection import train_test_split
# Create training/testing split
train, test = train_test_split(cv_data, test_size=0.3)

# Build spacy docs
file = open('/content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/error.txt', 'w')

db = get_spacy_docs(file, train)
db.to_disk('/content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/train_data.spacy')

db = get_spacy_docs(file, test)
db.to_disk('/content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/test_data.spacy')

file.close()


100%|██████████| 60/60 [00:00<00:00, 103.31it/s]


In [ ]:
!python -m spacy train /content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/config.cfg \
--output /content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/output \
--paths.train /content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/train_data.spacy \
--paths.dev /content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/training/test_data.spacy \
--gpu-id 0


ℹ Saving to output directory:
/content/drive/MyDrive/CV-Parsing-using-Spacy-3-master/data/output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 178kB/s]
config.json: 100% 481/481 [00:00<00:00, 3.79MB/s]
vocab.json: 899kB [00:00, 5.58MB/s]
merges.txt: 456kB [00:00, 6.30MB/s]
tokenizer.json: 1.36MB [00:00, 7.52MB/s]
2026-04-02 17:50:49.800499: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775152249.830774    6172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775152249.842436    6172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775152249

In [12]:
import json

doc = nlp(text)

# Create a dictionary to store the extracted data
extracted_data = {}

for ent in doc.ents:
    label = ent.label_
    value = ent.text.strip()

    # Since some labels appear multiple times, we store them in a list!
    if label not in extracted_data:
        extracted_data[label] = []

    if label == "Skills":
      skills = value.split(',')
      for skill in skills:
        extracted_data[label].append(skill.strip())
    else:
      extracted_data[label].append(value)

# Convert the dictionary to a beautifully formatted JSON string
json_output = json.dumps(extracted_data, indent=4, ensure_ascii=False)

print(json_output)



{
    "Name": [
        "Hoang Minh"
    ],
    "Designation": [
        "Thai (+84)818380406 | thaihm2014@gmail.com | https://www.linkedin.com/in/mean-thais/ EDUCATION University of Information Technology",
        "AI Engineer"
    ],
    "Degree": [
        "Bachelor of Computer Science"
    ],
    "Skills": [
        "Languages Python",
        "C++/C#",
        "Typescript",
        "Javascript. : Backend RESTful APIs",
        "FastAPI",
        "FastMCP",
        "Express",
        "Django",
        "Flask. : Databases MongoDB",
        "PostgreSQL",
        "Qdrant",
        "Milvus",
        "ElasticSearch. : Tools Docker",
        "CUDA",
        "Conda",
        "Nginx. : Frameworks Pytorch",
        "HuggingFace",
        "Transformers",
        "Langchain",
        "LangGraph",
        "LlamaIndex",
        "TensorFlow",
        "Pandas",
        "Numpy"
    ]
}
